In [1]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-32B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading Qwen3-32B model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Model loaded successfully!")
print("GPU available:", torch.cuda.is_available())


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer...
Loading Qwen3-32B model...


Loading weights: 100%|██████████| 707/707 [00:08<00:00, 82.99it/s]


Model loaded successfully!
GPU available: True


In [2]:

import json
import os

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

JSON_FILE_PATH = "datasets/aqua dataset/aqua_train_converted.json"

OUTPUT_FILE_PATH = "rehancot/reasoning dataset/first_10_cot_results.json"

# --------------------------------------------------
# 2. LOAD THE DATASET
# --------------------------------------------------

with open(JSON_FILE_PATH, "r", encoding="utf-8") as file:
    dataset = json.load(file)

if isinstance(dataset, dict):
    for key in ["data", "train", "examples", "questions"]:
        if key in dataset and isinstance(dataset[key], list):
            dataset = dataset[key]
            break

if not isinstance(dataset, list):
    raise ValueError("Expected a JSON list of questions.")

first_10 = dataset[:10]

print(f"Loaded {len(first_10)} questions.")

# Ensure the output directory exists.
os.makedirs(os.path.dirname(OUTPUT_FILE_PATH), exist_ok=True)

# --------------------------------------------------
# 3. EXTRACT QUESTION AND OPTIONS
# --------------------------------------------------

def get_question_text(item):
    question = item["question"]
    options = item["options"]

    if isinstance(options, list):
        options_text = "\n".join(
            f"{chr(65 + i)}. {option}"
            for i, option in enumerate(options)
        )
    else:
        options_text = str(options)

    return question, options_text

# --------------------------------------------------
# 4. GENERATE CHAIN OF THOUGHT
# --------------------------------------------------

results = []

for index, item in enumerate(first_10, start=1):

    question, options_text = get_question_text(item)

    prompt = f"""Solve the following question using concise,
logically complete step-by-step reasoning.

Question:
{question}

Options:
{options_text}

Explain the essential reasoning steps and then provide
your final answer. Avoid unnecessary filler.
"""

    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    print(f"\n{'=' * 50}")
    print(f"Question {index}/{len(first_10)}")
    print(question)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=True,
            temperature=0.6,
            top_p=0.95
        )

    # Keep only newly generated tokens.
    generated_ids = output_ids[0][
        inputs["input_ids"].shape[1]:
    ]

    # Separate reasoning from the final answer.
    token_ids = generated_ids.tolist()
    end_think_id = tokenizer.convert_tokens_to_ids("</think>")

    if end_think_id in token_ids:
        think_end_index = len(token_ids) - token_ids[::-1].index(
            end_think_id
        )

        reasoning = tokenizer.decode(
            token_ids[:think_end_index - 1],
            skip_special_tokens=True
        ).strip()

        final_answer = tokenizer.decode(
            token_ids[think_end_index:],
            skip_special_tokens=True
        ).strip()
    else:
        # Thinking may have been truncated by the token limit.
        reasoning = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        final_answer = ""

    print("\nREASONING:")
    print(reasoning)

    print("\nGENERATED ANSWER:")
    print(final_answer or "[No separate final answer generated]")

    results.append({
        "id": item.get("id", index),
        "question": question,
        "options": options_text,
        "reasoning": reasoning,
        "generated_answer": final_answer
    })

    # Save after every question.
    with open(OUTPUT_FILE_PATH, "w", encoding="utf-8") as file:
        json.dump(results, file, indent=2, ensure_ascii=False)

    print(f"Saved {len(results)} result(s).")

print(f"\nFinished! Results saved to: {OUTPUT_FILE_PATH}")


Loaded 10 questions.

Question 1/10
Two friends plan to walk along a 43-km trail, starting at opposite ends of the trail at the same time. If Friend P's rate is 15% faster than Friend Q's, how many kilometers will Friend P have walked when they pass each other?

REASONING:
<think>
Okay, let's see. So the problem is about two friends walking towards each other on a 43-km trail. They start at the same time from opposite ends. Friend P is 15% faster than Friend Q. We need to find out how many kilometers P has walked when they meet.

Hmm. Alright, let's break this down. First, since they're moving towards each other, their speeds add up. Let me denote Q's speed as some value, then P's speed would be 15% more. Let me assign variables. Let's say Q's speed is v km/h. Then P's speed is v + 0.15v = 1.15v km/h. 

The total distance between them is 43 km. When they meet, the sum of the distances they've each walked should equal 43 km. Also, they started at the same time and met after the same amo

In [2]:

import json
import os

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

JSON_FILE_PATH = "datasets/aqua dataset/aqua_train_converted.json"

OUTPUT_FILE_PATH = "rehancot/reasoning dataset/first_10_cot_results.json"

# --------------------------------------------------
# 2. LOAD THE DATASET
# --------------------------------------------------

with open(JSON_FILE_PATH, "r", encoding="utf-8") as file:
    dataset = json.load(file)

if isinstance(dataset, dict):
    for key in ["data", "train", "examples", "questions"]:
        if key in dataset and isinstance(dataset[key], list):
            dataset = dataset[key]
            break

if not isinstance(dataset, list):
    raise ValueError("Expected a JSON list of questions.")

first_10 = dataset[:10]

print(f"Loaded {len(first_10)} questions.")

# Ensure the output directory exists.
os.makedirs(os.path.dirname(OUTPUT_FILE_PATH), exist_ok=True)

# --------------------------------------------------
# 3. EXTRACT QUESTION AND OPTIONS
# --------------------------------------------------

def get_question_text(item):
    question = item["question"]
    options = item["options"]

    if isinstance(options, list):
        options_text = "\n".join(
            f"{chr(65 + i)}. {option}"
            for i, option in enumerate(options)
        )
    else:
        options_text = str(options)

    return question, options_text

# --------------------------------------------------
# 4. GENERATE CHAIN OF THOUGHT
# --------------------------------------------------

results = []

for index, item in enumerate(first_10, start=1):

    question, options_text = get_question_text(item)

    prompt = f"""Solve the following question using concise,
logically complete step-by-step reasoning.

Question:
{question}

Options:
{options_text}

Explain the essential reasoning steps and then provide
your final answer. Avoid unnecessary filler.
"""

    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    print(f"\n{'=' * 50}")
    print(f"Question {index}/{len(first_10)}")
    print(question)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=True,
            temperature=0.6,
            top_p=0.95
        )

    # Keep only newly generated tokens.
    generated_ids = output_ids[0][
        inputs["input_ids"].shape[1]:
    ]

    # Separate reasoning from the final answer.
    token_ids = generated_ids.tolist()
    end_think_id = tokenizer.convert_tokens_to_ids("</think>")

    if end_think_id in token_ids:
        think_end_index = len(token_ids) - token_ids[::-1].index(
            end_think_id
        )

        reasoning = tokenizer.decode(
            token_ids[:think_end_index - 1],
            skip_special_tokens=True
        ).strip()

        final_answer = tokenizer.decode(
            token_ids[think_end_index:],
            skip_special_tokens=True
        ).strip()
    else:
        # Thinking may have been truncated by the token limit.
        reasoning = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        final_answer = ""

    print("\nREASONING:")
    print(reasoning)

    print("\nGENERATED ANSWER:")
    print(final_answer or "[No separate final answer generated]")

    results.append({
        "id": item.get("id", index),
        "question": question,
        "options": options_text,
        "reasoning": reasoning,
        "generated_answer": final_answer
    })

    # Save after every question.
    with open(OUTPUT_FILE_PATH, "w", encoding="utf-8") as file:
        json.dump(results, file, indent=2, ensure_ascii=False)

    print(f"Saved {len(results)} result(s).")

print(f"\nFinished! Results saved to: {OUTPUT_FILE_PATH}")


Loaded 10 questions.

Question 1/10
Two friends plan to walk along a 43-km trail, starting at opposite ends of the trail at the same time. If Friend P's rate is 15% faster than Friend Q's, how many kilometers will Friend P have walked when they pass each other?

REASONING:
<think>
Okay, let's see. So the problem is about two friends walking towards each other on a 43-km trail. They start at the same time from opposite ends. Friend P is 15% faster than Friend Q. We need to find out how many kilometers P has walked when they meet.

Hmm. Alright, let's break this down. First, since they're moving towards each other, their speeds add up. Let me denote Q's speed as some value, then P's speed would be 15% more. Let me assign variables. Let's say Q's speed is v km/h. Then P's speed is v + 0.15v = 1.15v km/h. 

The total distance between them is 43 km. When they meet, the sum of the distances they've each walked should equal 43 km. Also, they started at the same time and met after the same amo

In [1]:
%pip install transformers accelerate --no-deps

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/dill-0.3.9-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.25a0+6627725-py3.12-linux-x86_64.egg is deprecated. pip 25.1 wi

In [ ]:
import os
from huggingface_hub import constants

print("Hugging Face cache:", constants.HF_HUB_CACHE)

print("\nQwen3 model folders:")
cache_path = constants.HF_HUB_CACHE

for folder in os.listdir(cache_path):
    if "Qwen3-32B" in folder:
        print(folder)